In [10]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Ensure reproducibility
np.random.seed(42)
print("Libraries imported successfully.")

Libraries imported successfully.


In [11]:
# --- CELL 2: Generate Raw E-Commerce Dataset with Intentional Messiness ---

# 1. Regions Table
regions_data = {
    'RegionID': ['REG-01', 'REG-02', 'REG-03', 'REG-04', 'REG-05'],
    'RegionName': ['North', 'South', 'East', 'West', 'Central'],
    'Country': ['India', 'India', 'India', 'India', 'India']
}
df_regions = pd.DataFrame(regions_data)

# 2. Customers Table (~1000 records)
n_customers = 1000
customer_ids = [f"CUST-{1001 + i}" for i in range(n_customers)]
names = [f"Customer_{i}" for i in range(n_customers)]
emails = [f"customer_{i}@example.com" for i in range(n_customers)]
join_dates = [pd.Timestamp('2024-01-01') + pd.to_timedelta(np.random.randint(0, 700), unit='D') for _ in range(n_customers)]
region_ids = np.random.choice(df_regions['RegionID'], n_customers)
ages = np.random.randint(18, 70, n_customers).astype(float) # Float to allow NaNs later
genders = np.random.choice(['Male', 'Female', 'Other'], n_customers, p=[0.52, 0.46, 0.02])

df_customers = pd.DataFrame({
    'CustomerID': customer_ids,
    'CustomerName': names,
    'Email': emails,
    'JoinDate': join_dates,
    'RegionID': region_ids,
    'Age': ages,
    'Gender': genders
})

# 3. Products Table (~50 records)
categories = ['Electronics', 'Apparel', 'Home & Kitchen', 'Sports']
sub_categories = {
    'Electronics': ['Smartphones', 'Laptops', 'Accessories'],
    'Apparel': ['Men Wear', 'Women Wear', 'Footwear'],
    'Home & Kitchen': ['Cookware', 'Decor', 'Appliances'],
    'Sports': ['Fitness', 'Outdoor', 'Team Sports']
}

product_ids = [f"PROD-{501 + i}" for i in range(50)]
product_names = [f"Product_{i}" for i in range(50)]
prod_categories = np.random.choice(categories, 50)
prod_subcategories = [np.random.choice(sub_categories[cat]) for cat in prod_categories]
unit_prices = np.random.uniform(100, 25000, 50).round(2)
cost_prices = (unit_prices * np.random.uniform(0.6, 0.85, 50)).round(2)

df_products = pd.DataFrame({
    'ProductID': product_ids,
    'ProductName': product_names,
    'Category': prod_categories,
    'SubCategory': prod_subcategories,
    'UnitPrice': unit_prices,
    'CostPrice': cost_prices
})

# 4. Orders Table (~5000 records)
n_orders = 5000
order_ids = [f"ORD-2026-{10000 + i}" for i in range(n_orders)]
order_custs = np.random.choice(df_customers['CustomerID'], n_orders)
order_dates = [pd.Timestamp('2025-01-01') + pd.to_timedelta(np.random.randint(0, 600), unit='D') for _ in range(n_orders)]
ship_dates = [od + pd.to_timedelta(np.random.randint(1, 7), unit='D') for od in order_dates]
statuses = np.random.choice(['Delivered', 'Shipped', 'Processing', 'Cancelled'], n_orders, p=[0.80, 0.10, 0.05, 0.05])
payment_methods = np.random.choice(['Credit Card', 'UPI', 'Net Banking', 'Cash on Delivery'], n_orders)

df_orders = pd.DataFrame({
    'OrderID': order_ids,
    'CustomerID': order_custs,
    'OrderDate': order_dates,
    'ShipDate': ship_dates,
    'OrderStatus': statuses,
    'PaymentMethod': payment_methods
})

# 5. Order Items Table (~10000 records)
n_items = 10000
item_ids = [f"ITEM-{100000 + i}" for i in range(n_items)]
item_orders = np.random.choice(df_orders['OrderID'], n_items)
item_prods = np.random.choice(df_products['ProductID'], n_items)
quantities = np.random.randint(1, 5, n_items)
discounts = np.random.choice([0.0, 0.05, 0.10, 0.15, 0.20], n_items, p=[0.5, 0.2, 0.15, 0.1, 0.05])

df_order_items = pd.DataFrame({
    'OrderItemID': item_ids,
    'OrderID': item_orders,
    'ProductID': item_prods,
    'Quantity': quantities,
    'Discount': discounts
})

# --- INJECTING INTENTIONAL MESSY DATA FOR CLEANING DEMO ---
# 1. Missing values in Customers Age
df_customers.loc[np.random.choice(df_customers.index, 50), 'Age'] = np.nan

# 2. Missing values in Orders ShipDate
df_orders.loc[np.random.choice(df_orders.index, 30), 'ShipDate'] = np.nan

# 3. Duplicate rows in Order Items
df_order_items = pd.concat([df_order_items, df_order_items.sample(50)], ignore_index=True)

# 4. Whitespace messiness in categories
df_products['Category'] = df_products['Category'].apply(lambda x: f"  {x} " if np.random.rand() > 0.7 else x)

# Save raw files (ensure your 'data/' folder exists or adjust path if saving in the current working directory)
df_regions.to_csv('../data/raw_regions.csv', index=False)
df_customers.to_csv('../data/raw_customers.csv', index=False)
df_products.to_csv('../data/raw_products.csv', index=False)
df_orders.to_csv('../data/raw_orders.csv', index=False)
df_order_items.to_csv('../data/raw_order_items.csv', index=False)

print("Raw datasets generated and exported successfully with intentional anomalies!")

Raw datasets generated and exported successfully with intentional anomalies!


In [12]:
print(df_regions.shape)
print(df_customers.shape)
print(df_products.shape)
print(df_orders.shape)
print(df_order_items.shape)

(5, 3)
(1000, 7)
(50, 6)
(5000, 6)
(10050, 5)


In [13]:
print("Missing customer ages:", df_customers['Age'].isna().sum())
print("Missing ship dates:", df_orders['ShipDate'].isna().sum())
print("Duplicate order items:", df_order_items.duplicated().sum())

Missing customer ages: 46
Missing ship dates: 30
Duplicate order items: 50


In [14]:
# Load raw data
df_customers = pd.read_csv('../data/raw_customers.csv')
df_products = pd.read_csv('../data/raw_products.csv')
df_orders = pd.read_csv('../data/raw_orders.csv')
df_order_items = pd.read_csv('../data/raw_order_items.csv')

# 1. Handle Missing Values
df_customers['Age'] = df_customers['Age'].fillna(df_customers['Age'].median())
df_orders['ShipDate'] = pd.to_datetime(df_orders['ShipDate'])
df_orders['OrderDate'] = pd.to_datetime(df_orders['OrderDate'])
df_orders['ShipDate'] = df_orders['ShipDate'].fillna(df_orders['OrderDate'] + pd.Timedelta(days=3))

# 2. Text Normalization & Trimming
df_customers['CustomerName'] = df_customers['CustomerName'].str.strip().str.title()
df_products['Category'] = df_products['Category'].str.strip()

# 3. Drop Duplicates
initial_count = len(df_order_items)
df_order_items = df_order_items.drop_duplicates()
print(f"Dropped {initial_count - len(df_order_items)} duplicate rows.")

Dropped 50 duplicate rows.


In [18]:
# --- CELL 4: Feature Engineering & Merging ---

# Merge Order Items with Products to calculate financial metrics
df_merged = df_order_items.merge(df_products[['ProductID', 'UnitPrice', 'CostPrice']], on='ProductID', how='left')

# Feature: Line Total and Line Profit
df_merged['LineTotal'] = (df_merged['Quantity'] * df_merged['UnitPrice'] * (1 - df_merged['Discount'])).round(2)
df_merged['LineProfit'] = (df_merged['Quantity'] * (df_merged['UnitPrice'] - df_merged['CostPrice']) * (1 - df_merged['Discount'])).round(2)

# Feature in Orders: Delivery Days
df_orders['DeliveryDays'] = (df_orders['ShipDate'] - df_orders['OrderDate']).dt.days

# Feature in Customers: Age Group Segmentation
bins = [0, 25, 35, 50, 100]
labels = ['18-25', '26-35', '36-50', '50+']
df_customers['AgeGroup'] = pd.cut(df_customers['Age'], bins=bins, labels=labels, right=False)

# Assign the final cleaned order items dataframe
df_order_items_cleaned = df_merged[['OrderItemID', 'OrderID', 'ProductID', 'Quantity', 'Discount', 'LineTotal', 'LineProfit']]

print("Feature engineering completed successfully!")

Feature engineering completed successfully!


In [19]:
# --- CELL 5: Export Cleaned Datasets ---
df_regions.to_csv('../data/cleaned_regions.csv', index=False)
df_customers.to_csv('../data/cleaned_customers.csv', index=False)
df_products.to_csv('../data/cleaned_products.csv', index=False)
df_orders.to_csv('../data/cleaned_orders.csv', index=False)
df_order_items_cleaned.to_csv('../data/cleaned_order_items.csv', index=False)

print("All cleaned datasets exported to data/ folder successfully!")

All cleaned datasets exported to data/ folder successfully!
